# Un model de limbaj de la zero (mvGPT)

Construim și antrenăm de la zero un model de tip GPT cu 14,6 milioane de parametri, de la date și tokenizare, prin atenție, preantrenare și ajustare fină supervizată (SFT), până la alinierea cu DPO și cu GRPO.

Notebookul rulează scripturile laboratorului [Un model de limbaj de la zero](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro), unde este explicat codul fiecărui fișier. Preantrenarea, ajustarea fină și GRPO durează câte 10 minute, iar DPO, 2 minute, indiferent de placa grafică. O placă grafică mai rapidă face mai mulți pași în acest timp, deci modelul iese mai bun.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

## Codul laboratorului

Fișierele se descarcă de pe site, ca în varianta cu `curl` de pe [pagina de introducere](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/prezentare/#pregătirea-mediului-de-lucru). Colab are deja PyTorch, NumPy și TensorBoard, așa că instalarea pachetelor din `requirements.txt` este rapidă.

In [ ]:
%%bash
mkdir -p mvgpt && cd mvgpt
for f in tokenizer.py data.py attention.py model.py checkpoint.py pretrain.py sample.py attention_map.py \
         sft.py reward.py preferences.py dpo.py grpo.py requirements.txt; do
  curl -fsSO "https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/cod/$f"
done
ls

In [ ]:
%cd mvgpt
%pip install -q -r requirements.txt

## [Datele și tokenizarea](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/date-si-tokenizare/)

Scriptul descarcă primii 80 MB din fiecare fișier al seturilor TinyStories și TinyStories-Instruct, construiește vocabularul de 10.000 de cuvinte și salvează textele ca tokenuri.

In [ ]:
!python data.py

## [Atenția, scrisă de mână](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/atentia-scrisa-de-mana/)

Atenția cauzală, calculată de mână pe o propoziție de șase tokenuri și comparată cu funcția din PyTorch, apoi stratul de atenție cu șase capete al modelului.

In [ ]:
!python attention.py

## [Modelul GPT](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/modelul-gpt/)

Numărul de parametri ai modelului și pierderea pe tokenuri aleatoare, înainte de antrenare.

In [ ]:
!python model.py

## [Preantrenarea](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/preantrenare/)

Graficele antrenării se pot urmări în TensorBoard. Celula următoare îl pornește, iar graficele se actualizează în timp ce modelul se antrenează.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
!python pretrain.py 10

## [Generarea textului](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/generarea-textului/)

Două povești scrise de modelul preantrenat, apoi tokenul la care se uită cel mai mult fiecare cap de atenție, de la ultimul token al propoziției „Tom and Lily went to the park. Tom gave the ball to”.

In [ ]:
!python sample.py checkpoints/pretrain/model.pt "Once upon a time"

In [ ]:
!python attention_map.py checkpoints/pretrain/model.pt

## [Ajustarea fină supervizată](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/ajustare-fina-supervizata/)

Modelul preantrenat învață să scrie o poveste care conține cuvintele cerute.

In [ ]:
!python sft.py 10

In [ ]:
!python sample.py checkpoints/sft/model.pt "Words: rain, cake, jump Story:"

## [Recompensa și DPO](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/recompensa-si-dpo/)

Recompensa este proporția de cuvinte cerute care apar în poveste. Prima celulă măsoară modelul preantrenat și pe cel ajustat, pe 50 de instrucțiuni de validare.

In [ ]:
!python reward.py checkpoints/pretrain/model.pt checkpoints/sft/model.pt

Modelul ajustat scrie patru povești pentru fiecare instrucțiune, iar povestea cu cele mai multe cuvinte cerute și cea cu cele mai puține formează o pereche de preferințe. Scriptul nu are un buget de timp. Pe o placă grafică, celula următoare scrie 500 de perechi de antrenare și 50 de validare, ca pe pagina laboratorului, în circa 2.700 de povești. Pe procesor, aceleași povești ar cere câteva ore, așa că celula scrie doar 32 de perechi de antrenare și 8 de validare. Cu atât de puține perechi, DPO învață mult mai puțin.

In [ ]:
import torch

# Numărul de perechi de antrenare și de validare: valorile implicite pe o placă grafică, mai puține pe procesor.
perechi = "500 50" if torch.cuda.is_available() else "32 8"
!python preferences.py {perechi}

DPO învață din perechi timp de 2 minute, față de o copie înghețată a modelului ajustat. Apoi măsurăm modelul obținut. Înaintea fiecărui model, `reward.py` pornește generatorul de numere aleatoare de la aceeași valoare, așa că rezultatul se poate compara direct cu cele de mai sus.

In [ ]:
!python dpo.py 2

In [ ]:
!python reward.py checkpoints/dpo/model.pt

## [Învățare prin consolidare cu GRPO](https://inteligentartificiala.ro/laboratoare/model-lingvistic-de-la-zero/invatare-prin-consolidare-cu-grpo/)

Modelul ajustat scrie, la fiecare rundă, câte 8 povești pentru 16 instrucțiuni și învață din propriile povești: poveștile mai bune decât media grupului devin mai probabile. Pe procesor, o singură rundă poate depăși cele 10 minute, iar scriptul se oprește după prima rundă.

In [ ]:
!python grpo.py 10

In [ ]:
!python reward.py checkpoints/grpo/model.pt